## 1. Setup

In [5]:
import os, sys, subprocess
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import torch
from datetime import datetime
import logging
import warnings
warnings.filterwarnings('ignore')

def log(msg=""):
    """Lightweight logger that also supports blank lines."""
    print(msg)

# Suppress warnings
os.environ['WANDB_DISABLED'] = 'true'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print("✅ Imports complete")

✅ Imports complete


## 2. Configuration

In [6]:
# --- CONFIGURATION (V3 Speed - Aligned with Pre-generated Pairs + FP16) ---
CONFIG = {
    # Model
    'model_name': 'nomic-ai/nomic-embed-text-v1.5',
    'output_dir': 'models/real_servicenow_finetuned_nomic_v3_speed',
    
    # Pre-generated pairs (validated)
    'train_pairs_path': 'data_new/fixed_training_pairs.json',
    'eval_pairs_path': 'data_new/fixed_test_pairs.json',
    
    # Training hyperparameters
    'epochs': 4,
    'batch_size': 16,
    'lr': 2e-5,
    'max_seq_length': 512,
    'warmup_ratio': 0.1,
    
    # Speed optimizations
    'use_fp16': True,         # FP16 for speed (Speed version)
    'dataloader_workers': 0,  # 0 for MPS/CPU compatibility
    
    # Seed
    'seed': 42
}

# Set seeds
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])

# Device detection
DEVICE = 'cpu'
if torch.cuda.is_available():
    # Pick CUDA device with most free memory
    free_totals = [torch.cuda.mem_get_info(i) for i in range(torch.cuda.device_count())]
    best_idx = max(range(len(free_totals)), key=lambda i: free_totals[i][0])
    torch.cuda.set_device(best_idx)
    DEVICE = f'cuda:{best_idx}'
    torch.cuda.manual_seed_all(CONFIG['seed'])
    name = torch.cuda.get_device_name(best_idx)
    total_mem_gb = torch.cuda.get_device_properties(best_idx).total_memory / 1024**3
    free_gb = free_totals[best_idx][0] / 1024**3
    
    # Dynamic batch sizing for memory-limited GPUs
    if total_mem_gb >= 20:
        CONFIG['batch_size'] = 32
    elif total_mem_gb >= 12:
        CONFIG['batch_size'] = 16
    elif total_mem_gb >= 8:
        CONFIG['batch_size'] = 8
    else:
        CONFIG['batch_size'] = 4
    
    # Reduce sequence length on smaller GPUs
    if total_mem_gb < 12:
        CONFIG['max_seq_length'] = min(CONFIG['max_seq_length'], 256)
    
    log(f"🚀 CUDA Detected ({name}) | Total: {total_mem_gb:.1f}GB | Free: {free_gb:.1f}GB → batch_size={CONFIG['batch_size']}, max_seq_len={CONFIG['max_seq_length']}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
    torch.mps.manual_seed(CONFIG['seed'])
    log("🍎 MPS (Apple Silicon) Detected")
    CONFIG['batch_size'] = 8
    CONFIG['use_fp16'] = False
    CONFIG['max_seq_length'] = min(CONFIG['max_seq_length'], 256)
else:
    DEVICE = 'cpu'
    log("⚠️  CPU Mode")
    CONFIG['batch_size'] = 8
    CONFIG['use_fp16'] = False
    CONFIG['max_seq_length'] = min(CONFIG['max_seq_length'], 256)

log(f"📊 Device: {DEVICE}, Batch Size: {CONFIG['batch_size']}, FP16: {CONFIG['use_fp16']}")

🚀 CUDA Detected (NVIDIA GeForce RTX 2080 SUPER) | Total: 8.0GB | Free: 3.9GB → batch_size=4, max_seq_len=256
📊 Device: cuda:0, Batch Size: 4, FP16: True


## 3. Load Pre-Generated Pairs

In [7]:
from sentence_transformers import InputExample

def load_pairs_from_json(pairs_path):
    """Load pre-generated pairs from JSON."""
    pairs_file = Path(pairs_path)
    
    if not pairs_file.exists():
        raise FileNotFoundError(f"Pairs file not found: {pairs_file}")
    
    log(f"📂 Loading pairs from: {pairs_file}")
    
    with open(pairs_file, 'r') as f:
        data = json.load(f)
    
    texts1 = data['texts1']
    texts2 = data['texts2']
    labels = data['labels']
    metadata = data.get('metadata', {})
    
    log(f"✅ Loaded {len(labels):,} pairs")
    log(f"   Positives: {sum(labels):,} ({100*sum(labels)/len(labels):.1f}%)")
    log(f"   Negatives: {len(labels) - sum(labels):,} ({100*(len(labels)-sum(labels))/len(labels):.1f}%)")
    
    # Convert to InputExample format
    pairs = []
    for i in range(len(labels)):
        pairs.append(InputExample(
            texts=[texts1[i], texts2[i]],
            label=float(labels[i])
        ))
    
    return pairs, metadata

# Load training and eval pairs
log(f"\n{'='*80}")
log("LOADING PRE-GENERATED PAIRS")
log(f"{'='*80}")

train_pairs, train_metadata = load_pairs_from_json(CONFIG['train_pairs_path'])
log()
eval_pairs, eval_metadata = load_pairs_from_json(CONFIG['eval_pairs_path'])

log(f"\n📊 Summary:")
log(f"   Training pairs: {len(train_pairs):,}")
log(f"   Eval pairs: {len(eval_pairs):,}")
log(f"   Baseline separability: {train_metadata.get('baseline_separability', 'N/A')}")


LOADING PRE-GENERATED PAIRS
📂 Loading pairs from: data_new\fixed_training_pairs.json
✅ Loaded 10,000 pairs
   Positives: 5,000 (50.0%)
   Negatives: 5,000 (50.0%)

📂 Loading pairs from: data_new\fixed_test_pairs.json
✅ Loaded 1,000 pairs
   Positives: 500 (50.0%)
   Negatives: 500 (50.0%)

📊 Summary:
   Training pairs: 10,000
   Eval pairs: 1,000
   Baseline separability: 0.37416520714759827


## 4. Training

In [8]:
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader

log(f"\n{'='*80}")
log("TRAINING SENTENCE TRANSFORMER")
log(f"{'='*80}")

# Load model with OOM fallback to CPU
log(f"\n📥 Loading model: {CONFIG['model_name']}")
try:
    model = SentenceTransformer(
        CONFIG['model_name'],
        device=DEVICE,
        trust_remote_code=True
)
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        log("   OOM on GPU during load → falling back to CPU")
        DEVICE = 'cpu'
        CONFIG['use_fp16'] = False
        model = SentenceTransformer(
	    CONFIG['model_name'],
            device=DEVICE,
            trust_remote_code=True
)
    else:
        raise

# Apply max sequence length constraint
try:
    model.max_seq_length = CONFIG['max_seq_length']
    log(f"   Applied max_seq_length={CONFIG['max_seq_length']}")
except Exception as e:
    log(f"   Warning: Could not set max_seq_length: {e}")

# Try enabling gradient checkpointing to save memory
try:
    first = model._first_module()
    auto_model = getattr(first, 'auto_model', None)
    if auto_model and hasattr(auto_model, 'gradient_checkpointing_enable'):
        auto_model.gradient_checkpointing_enable()
        log("   🧠 Enabled gradient checkpointing")
except Exception as e:
    log(f"   Warning: Gradient checkpointing not enabled: {e}")

# Create data loaders
train_dataloader = DataLoader(
    train_pairs,
    shuffle=True,
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['dataloader_workers'],
    pin_memory=DEVICE.startswith('cuda')
)

# Define loss
train_loss = losses.ContrastiveLoss(model)

# Setup training
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = Path(CONFIG['output_dir']) / f"v3_speed_{timestamp}"
save_path.mkdir(parents=True, exist_ok=True)

warmup_steps = int(len(train_dataloader) * CONFIG['epochs'] * CONFIG['warmup_ratio'])

log(f"\n📊 Training Setup:")
log(f"   Model: {CONFIG['model_name']}")
log(f"   Batches/epoch: {len(train_dataloader)}")
log(f"   Total training steps: {len(train_dataloader) * CONFIG['epochs']}")
log(f"   Warmup steps: {warmup_steps}")
log(f"   FP16: {CONFIG['use_fp16']}")
log(f"   Output: {save_path}")

# Clear any stale CUDA memory before training
try:
    if DEVICE.startswith('cuda'):
        torch.cuda.empty_cache()
        log("   Freed CUDA cache before training")
except Exception:
    pass

# Train (no mid-epoch evaluation to avoid ValueError when no evaluator provided)
log(f"\n🚀 Starting Training ({CONFIG['epochs']} epochs)...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=CONFIG['epochs'],
    warmup_steps=warmup_steps,
    optimizer_params={'lr': CONFIG['lr']},
    output_path=str(save_path),
    use_amp=CONFIG['use_fp16'],
    show_progress_bar=True
)

log(f"\n✅ Training complete!")
log(f"💾 Model saved to: {save_path}")

# Load best model with trust_remote_code and safe fallback
try:
    model = SentenceTransformer(str(save_path), device=DEVICE, trust_remote_code=True)
except Exception as e:
    log(f"   Reload failed on {DEVICE}: {e} → trying CPU")
    model = SentenceTransformer(str(save_path), device='cpu', trust_remote_code=True)


TRAINING SENTENCE TRANSFORMER

📥 Loading model: nomic-ai/nomic-embed-text-v1.5


<All keys matched successfully>
You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


   Applied max_seq_length=256
   🧠 Enabled gradient checkpointing

📊 Training Setup:
   Model: nomic-ai/nomic-embed-text-v1.5
   Batches/epoch: 2500
   Total training steps: 10000
   Warmup steps: 1000
   FP16: True
   Output: models\real_servicenow_finetuned_nomic_v3_speed\v3_speed_20251219_163801
   Freed CUDA cache before training

🚀 Starting Training (4 epochs)...


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss
500,0.012500
1000,0.009000
1500,0.007700
2000,0.008000
2500,0.007100
3000,0.004100
3500,0.003900
4000,0.003500
4500,0.004400
5000,0.003700



✅ Training complete!
💾 Model saved to: models\real_servicenow_finetuned_nomic_v3_speed\v3_speed_20251219_163801


<All keys matched successfully>


## 5. Evaluation

In [9]:
from sklearn.metrics import roc_auc_score, f1_score
from scipy.stats import spearmanr, pearsonr

log(f"\n{'='*80}")
log("EVALUATION")
log(f"{'='*80}")

def evaluate_pairs(pairs, model, name="Test"):
    """Evaluate model on pairs."""
    texts1 = [ex.texts[0] for ex in pairs]
    texts2 = [ex.texts[1] for ex in pairs]
    labels = np.array([ex.label for ex in pairs])
    
    # Encode
    log(f"\n⏳ Encoding {len(pairs)} pairs...")
    emb1 = model.encode(texts1, batch_size=CONFIG['batch_size'], show_progress_bar=True, convert_to_numpy=True)
    emb2 = model.encode(texts2, batch_size=CONFIG['batch_size'], show_progress_bar=True, convert_to_numpy=True)
    
    # Cosine similarity
    norm1 = np.linalg.norm(emb1, axis=1, keepdims=True)
    norm2 = np.linalg.norm(emb2, axis=1, keepdims=True)
    scores = np.sum((emb1 / norm1) * (emb2 / norm2), axis=1)
    
    # Metrics
    spearman, _ = spearmanr(labels, scores)
    pearson, _ = pearsonr(labels, scores)
    auc = roc_auc_score(labels, scores)
    
    # F1 at threshold 0.5
    preds = (scores > 0.5).astype(int)
    f1 = f1_score(labels, preds)
    
    log(f"\n📊 {name} Results:")
    log(f"   Spearman correlation: {spearman:.4f}")
    log(f"   Pearson correlation:  {pearson:.4f}")
    log(f"   ROC-AUC: {auc:.4f}")
    log(f"   F1 (threshold=0.5): {f1:.4f}")
    
    return {'spearman': spearman, 'pearson': pearson, 'auc': auc, 'f1': f1}

# Evaluate
eval_results = evaluate_pairs(eval_pairs, model, "Eval Set")

log(f"\n✅ Evaluation complete!")


EVALUATION

⏳ Encoding 1000 pairs...


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 250/250 [00:06<00:00, 40.46it/s]


📊 Eval Set Results:
   Spearman correlation: 0.3736
   Pearson correlation:  0.3875
   ROC-AUC: 0.7157
   F1 (threshold=0.5): 0.6485

✅ Evaluation complete!
